# 1. Installation

In [2]:
# Run this cell first to install necessary libraries
%%capture
!pip install rouge_score
!pip install unsloth
# Update Unsloth to the latest nightly version for Llama 3.2 support
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

# 2. Imports and Configuration

In [3]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True # 4-bit quantization to reduce memory usage

# --- MODEL SELECTION ---
# I select the 3B model to balance performance (ROUGE > 0.205) and parameter count.
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# 3. Load Model and Tokenizer

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

# 4. Add LoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# 5. Data Preparation

In [6]:
# Load the dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split = "train")

# --- DATA SPLIT ---
# Split 95% for training and 5% for testing using the specified seed
dataset = dataset.train_test_split(test_size=0.05, seed=4016)
train_dataset = dataset['train']
test_dataset = dataset['test']

# Set up chat template (Llama-3 style)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# Data formatting function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["context"]
    outputs      = examples["response"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Format as a conversation
        # If context is present, we include it.
        # This matches standard Llama 3 prompt structure
        if input:
            instruction = f"{instruction}\n\nContext:\n{input}"

        conversation = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output},
        ]

        text = tokenizer.apply_chat_template(conversation, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts, }

# Apply formatting to training data
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/14260 [00:00<?, ? examples/s]

 # 6. Training Configuration

In [7]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # --- TRAINING DURATION ---
        # Changed from max_steps=60 to 1 full epoch to ensure sufficient learning for ROUGE score target
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Configure trainer to only train on responses (masking instructions)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/14260 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/14260 [00:00<?, ? examples/s]

# Model Evaluation


In [8]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,260 | Num Epochs = 1 | Total steps = 1,783
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yanmuhan990731 (yanmuhan990731-city-university-of-hong-kong) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
10,1.909900
20,1.862300
30,1.798200
40,1.698000
50,1.624200
60,1.802100
70,1.804400
80,1.492500
90,1.694000
100,1.863100


Unsloth: Will smartly offload gradients to save VRAM!


train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇█
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▂▁▇▄▃▇▅▅▃▃▁█▄▇▃▄▄▃▂▃▃▄▄▇▄▄▄▄▃▄▅▂▇▂▇▄▃▆▆▇
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁
train/loss,█▇▇▃▇▄▆▇▆▅▄▆▄█▅▇▄▄▅▅██▄▆▇▃▄▄▅▁▇▅▇▆▆▆▇▄▅▇
total_flos,6.904009191703757e+16
train/epoch,1
train/global_step,1783
train/grad_norm,0.60545
train/learning_rate,0.0
train/loss,1.8556


# 8. Evaluation

In [9]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def generate_response(instruction, context=""):
    if context:
        instruction = f"{instruction}\n\nContext:\n{context}"

    messages = [
        {"role": "user", "content": instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(input_ids = inputs, max_new_tokens = 128, use_cache = True, temperature=0.1)
    decoded_output = tokenizer.batch_decode(outputs)

    # Extract only the assistant's response
    # (Simple parsing logic for Llama 3 format)
    response = decoded_output[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", "").strip()
    return response

# Run evaluation on the test set
print("Starting Evaluation...")
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

# Using tqdm for progress bar
for item in tqdm(test_dataset):
    instruction = item['instruction']
    context = item['context']
    reference_response = item['response']

    # Generate response
    generated_response = generate_response(instruction, context)

    # Calculate scores
    scores = scorer.score(reference_response, generated_response)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

# Calculate and print averages
print(f"\nEvaluation Results for {model_name}:")
print(f"Average ROUGE-1: {np.mean(rouge1_scores):.4f}")
print(f"Average ROUGE-2: {np.mean(rouge2_scores):.4f}")
print(f"Average ROUGE-L: {np.mean(rougeL_scores):.4f}")

if np.mean(rougeL_scores) >= 0.205:
    print("Success: ROUGE-L threshold met!")
else:
    print("Threshold not met. Consider switching to the 8B model or increasing epochs.")

Starting Evaluation...


 16%|█▌        | 122/751 [07:25<51:15,  4.89s/it]Unsloth: Input IDs of shape torch.Size([1, 2364]) with length 2364 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
 41%|████      | 308/751 [18:22<36:22,  4.93s/it]Unsloth: Input IDs of shape torch.Size([1, 4861]) with length 4861 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
100%|██████████| 751/751 [43:35<00:00,  3.48s/it]


Evaluation Results for unsloth/Llama-3.2-3B-Instruct-bnb-4bit:
Average ROUGE-1: 0.4353
Average ROUGE-2: 0.2543
Average ROUGE-L: 0.3639
Success: ROUGE-L threshold met!
